# Stage buckets and baseline evidence

Which stage information explains BST? What remains for correction model? Current chains include later-added evolutions. All medians here describe input population; training must fit its own baselines.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pokeml.eda.snapshot import load_analysis_snapshot

df = load_analysis_snapshot()
OUTPUT = Path("artifacts/eda/01_stage_analysis")
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Validated analysis snapshot: {len(df):,} rows")

Validated analysis snapshot: 1,082 rows


## Stage distributions

Boxes show middle 50%; orange lines show medians. Whiskers reach observations within 1.5 IQR; circles lie outside. Horizontal jitter separates overlaps without changing BST.

In [3]:
from pokeml.eda.stages import stage_summary, stage_outliers, stage_residuals, morphology_summary, project_to_mock_stage
from pokeml.visualisation.viz_eda.stage_buckets import plot_stage_buckets
summary = stage_summary(df)
display(summary.round(2))
summary.to_csv(OUTPUT / "stage_summary.csv", index=False)
fig, ax = plot_stage_buckets(df)
fig.savefig(OUTPUT / "stage_buckets.png", dpi=180, bbox_inches="tight")
plt.show()

,rarity,stage,count,median,mean,std,min,max,q1,q3,iqr,sparse
0,legendary,s1c2,3,385.0,406.33,118.45,300,534,342.50,459.5,117.00,True
1,legendary,s1c3,1,200.0,200.00,NaN,200,200,200.00,200.0,0.00,True
2,legendary,s2c2,3,570.0,573.33,25.17,550,600,560.00,585.0,25.00,True
3,legendary,s2c3,1,400.0,400.00,NaN,400,400,400.00,400.0,0.00,True
4,legendary,s3c3,2,680.0,680.00,0.00,680,680,680.00,680.0,0.00,True
5,legendary,single,95,590.0,604.05,47.66,450,720,580.00,600.0,20.00,False
6,regular,s1c2,237,308.0,312.23,54.19,180,535,280.00,335.0,55.00,False
7,regular,s1c3,116,300.0,285.10,40.36,180,395,263.75,310.0,46.25,False
8,regular,s2c2,264,487.0,482.09,38.44,236,600,460.00,505.0,45.00,False
9,regular,s2c3,118,405.0,391.02,59.67,205,515,365.00,420.0,55.00,False


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16169/2860648861.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Named exceptions

600-BST end-of-chain archetypes overlap in original boxplot. Slaking is 670. Statistical outliers remain valid data; no hard cap or automatic exclusion.

In [4]:
outliers = stage_outliers(df)
display(outliers[["name", "stage", "total_stats", "generation", "side", "lower_fence", "upper_fence"]])
outliers.to_csv(OUTPUT / "stage_outliers.csv", index=False)
regular = df[df.rarity.eq("regular")]
sensitivity = pd.DataFrame({"all_median": regular.groupby("stage").total_stats.median(), "without_named_outliers_median": regular[~regular.id.isin(outliers.id)].groupby("stage").total_stats.median()})
display(sensitivity)
print("Sensitivity only: headline data retain every outlier")

,name,stage,total_stats,generation,side,lower_fence,upper_fence
0,sunkern,s1c2,180,2,lower,197.500,417.500
1,snom,s1c2,185,8,lower,197.500,417.500
2,kricketot,s1c2,194,4,lower,197.500,417.500
3,poipole,s1c2,420,7,upper,197.500,417.500
4,gligar,s1c2,430,2,upper,197.500,417.500
5,sneasel,s1c2,430,2,upper,197.500,417.500
6,sneasel_hisui,s1c2,430,8,upper,197.500,417.500
7,misdreavus,s1c2,435,2,upper,197.500,417.500
8,tangela,s1c2,435,1,upper,197.500,417.500
9,qwilfish_hisui,s1c2,440,8,upper,197.500,417.500


,all_median,without_named_outliers_median
stage,,
s1c2,308.0,305.0
s1c3,300.0,300.0
s2c2,487.0,487.5
s2c3,405.0,405.0
s3c3,523.0,520.0
single,478.0,484.0


Sensitivity only: headline data retain every outlier


## Starter design roles and form sensitivity

Starter status is separate from evolutionary position. Counts span nine generations; related evolution stages are not independent observations.

In [5]:
from pokeml.features.feature_initial import apply_initial_tag, get_initial_pokes
from pokeml.eda.power_creep import default_species
tagged = apply_initial_tag(df, get_initial_pokes())
starters = tagged[tagged.rarity.eq("regular")].groupby(["stage", "initial"], observed=True).total_stats.agg(["count", "median", "std"])
display(starters.round(2))
species_summary = stage_summary(default_species(df))
display(species_summary[species_summary.rarity.eq("regular")].merge(summary[summary.rarity.eq("regular")], on=["rarity", "stage"], suffixes=("_species", "_with_forms"))[["stage", "count_species", "count_with_forms", "median_species", "median_with_forms"]])

count  median    std
stage  initial                          
s1c2   non_initial    237   308.0  54.19
s1c3   initial         27   310.0   4.17
       non_initial     89   280.0  42.77
s2c2   non_initial    264   487.0  38.44
s2c3   initial         27   405.0   6.30
       non_initial     91   390.0  66.89
s3c3   initial         30   530.0   2.60
       non_initial     95   510.0  48.72
single non_initial    117   478.0  70.20

,stage,count_species,count_with_forms,median_species,median_with_forms
0,s1c2,218,237,308.0,308.0
1,s1c3,114,116,300.0,300.0
2,s2c2,245,264,485.0,487.0
3,s2c3,114,118,405.0,405.0
4,s3c3,119,125,520.0,523.0
5,single,115,117,478.0,478.0


## Original mock-stage projection

Original shape notebook subtracted stage median and added shared reference. Function now lives in package. This is descriptive centring, not an independently validated biological mechanism.

In [6]:
projected, stage_medians, shared_reference, legendary_reference = project_to_mock_stage(df)
display(projected[projected.rarity.eq("regular")].groupby("shape").bst_projected_mock_stage.agg(["count", "mean", "median", "std"]).round(2))
display(morphology_summary(df).round(2))

,count,mean,median,std
shape,,,,
armor,47,398.87,406.0,50.67
arms,40,427.00,427.5,18.63
ball,49,406.43,418.0,61.86
blob,43,403.70,418.0,54.93
bug-wings,22,401.41,398.0,84.19
fish,47,415.17,422.0,62.35
heads,22,431.55,428.0,40.04
humanoid,137,416.23,417.0,47.96
legs,30,418.03,417.0,58.25


,count,median_bst,median_residual,mean_residual,sparse
shape,,,,,
bug-wings,22,455.0,-27.0,-23.59,False
armor,47,330.0,-19.0,-26.13,False
wings,78,435.0,-13.5,-9.10,False
humanoid,137,455.0,-8.0,-8.77,False
legs,30,310.0,-8.0,-6.97,False
ball,49,340.0,-7.0,-18.57,False
blob,43,405.0,-7.0,-21.30,False
fish,47,450.0,-3.0,-9.83,False
squiggle,37,430.0,0.0,-9.16,False


## Remaining association with size

Spearman correlations examine monotonic association. Stage centring removes bucket median only; type, family and other confounding can remain.

In [7]:
residuals = stage_residuals(regular)
display(residuals[["height", "weight", "total_stats", "stage_residual"]].corr(method="spearman").round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
for ax, field in zip(axes, ["height", "weight"]):
    ax.scatter(residuals[field], residuals.stage_residual, s=12, alpha=.35)
    ax.set_xscale("log")
    ax.axhline(0, color="grey", linestyle="--")
    ax.set(xlabel=f"{field} ({"m" if field == "height" else "kg"}; log scale)", ylabel="BST minus stage median")
fig.savefig(OUTPUT / "size_stage_residuals.png", dpi=180)
plt.show()

,height,weight,total_stats,stage_residual
height,1.000,0.815,0.745,0.328
weight,0.815,1.000,0.664,0.354
total_stats,0.745,0.664,1.000,0.480
stage_residual,0.328,0.354,0.480,1.000


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16169/503895985.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Baseline recommendation

Stage medians are useful candidate baselines. Starter grouping and morphology are correction hypotheses. Bucket overlap, ambiguous chain length and wide single-stage spread require uncertainty. Next modelling step must compare fixed baseline, residual regressor and direct regressor on matching forward cohorts.

## Original author notes

Original comments preserved below. They document exploration, not validated rules. Audit sections above qualify or challenge them.

### dist_stage

```python
# Load the minimal_pkdx
# Define how to split each stage and rarity
# Get the data for each group, get also the median baseline and plot
```

### dist_shape

```python
# Stage medians for regular Pokémon
# Shared reference stage for regular Pokémon
# Legendary reference can be separate
# Regular: project to the common mock stage
# Legendary: project to its own common baseline, if you want it included
# Armor, blob, bug-wings, heads, legs
```

### Original exploratory notes

Historical comments and docstrings; current analysis above qualifies their claims.

```text
# Change to project root (parent of notebooks/)
# For imports from pokeml/
# Verify: /path/to/project_root
```